# PR14 · Remove a modeled nuisance contribution without declaring the residual pure

<!-- paper-first -->
### Research question

**Reading:** [PP02](../../curriculum/papers/processing.md#pp02), [PD01](../../curriculum/papers/design.md#pd01). Review the assigned figure or result before starting the lesson.

**Question:** What warrants calling a regressor nuisance, and what could remain after removing its fitted contribution?

Record a prediction, a source location, and one point you want this lesson to clarify. Ask your AI tutor to distinguish the paper’s evidence from its interpretation.
<!-- /paper-first -->

**Format:** 75–100 minutes for this lesson and its small executable lab, followed by the explicitly labeled upstream practice. Run this notebook from a fresh kernel, top to bottom. Core data are synthetic. Software: NumPy, SciPy, Matplotlib; extra imports are stated in code. This is one stage of a longer processing course, not a replacement for supervised research training.

## Learning objectives

- Verify regression as a projection and inspect rank.
- Demonstrate incompatible filtering and confound regression.
- Explain ICA component ambiguity and the separate need for artifact classification.

## Understand the operation

Nuisance regression subtracts variation represented by selected columns of a design matrix. Least squares estimates their coefficients; its residual is orthogonal to those included columns under the fitted inner product. That is a calculation check, not proof that all artifacts are removed. If a column overlaps a biological effect, that shared component may be removed too. Missing values, sample misalignment and duplicated columns have to be handled explicitly.

Filtering and regression interact because their projections need not commute. Filtering only the data and then regressing an unfiltered confound can reintroduce the very component removed by the filter. We use an exact slow-basis projection to demonstrate this algebra cleanly; a practical Butterworth/censoring pipeline has additional details. A coordinated implementation or a jointly specified model makes the intended operation inspectable.

Independent component analysis is another decomposition approach. It estimates sources and mixing under statistical assumptions rather than labeling sources as brain or noise. Component order and sign are arbitrary; a large component is not automatically an artifact. Spatial distribution, frequency content, relationships with motion and a justified classifier may inform decisions. Removing components changes the data according to that selection. The optional tiny ICA calculation here separates non-Gaussian toy sources and verifies correlation after accounting for permutation/sign. It does not implement ICA-AROMA, FIX or any clinical denoising standard. For real work, choose and document a strategy appropriate to the question and available acquisition information.

## Transformation contract

**Input:** matched time rows in signal and confound arrays. **Output:** coefficients and residuals; separately ICA sources/mixing. **Removed:** modeled subspace, regardless of biological origin. **Preserved:** complement under the chosen operation. **Check:** rank, sample identity, reconstruction/projection errors and residual behavior.


## Read the actual course material

- [DartBrains: Separating Signal from Noise with ICA](https://github.com/ljchang/dartbrains/blob/5d727f7a72a7f20bfb32603cac8509d78b9647f2/content/ICA.py). CC BY-SA 4.0; linked pinned chapter.
- [fMRIPrep: pipeline details](https://fmriprep.org/en/stable/workflows.html). Official project documentation; linked only.

Read the named topic alongside this lesson; compare its real-image assumptions with our controlled example. These notebooks use original explanations and original code, not copied upstream passages. The source chapter is the place to continue to a complete real-tool practical. External software and downloaded datasets are not silently run by this notebook.


## Predict, then ask your AI assistant

Use Goose with your installed Ollama model, or ChatGPT. The model is a tutor and code author; the local Python runtime performs these calculations. Paste:

> Derive the least-squares residual, show the confound/filter mismatch, and verify orthogonality. For the separate ICA example, account for sign/permutation when comparing components. Do not choose an artifact component solely by its index or variance. Return at most 20 executable lines per cell, show units and array shapes, and preserve the original. Explain the prediction before running. If an assertion fails, diagnose the disagreement rather than deleting the check.

Write your prediction before executing the reference cells below.


In [1]:
import numpy as np
from sklearn.decomposition import FastICA
n=300; t=np.arange(n)*2.
d=np.sin(2*np.pi*.005*t); q=np.sin(2*np.pi*.05*t); s=np.sin(2*np.pi*.08*t)
c=d+q; y=3*d+2*q+s
D=np.column_stack([np.ones(n),d])
def resid(v,X): return v-X@np.linalg.lstsq(X,v,rcond=None)[0]
yf=resid(y,D); cf=resid(c,D)
bad=resid(yf,np.column_stack([np.ones(n),c]))
C=np.column_stack([np.ones(n),cf]); good=resid(yf,C)
print('bad/good drift projection:',np.dot(d,bad)/np.dot(d,d),np.dot(d,good)/np.dot(d,d))
assert np.allclose(good,s,atol=1e-10)
assert np.isclose(np.dot(d,bad)/np.dot(d,d),-1)
assert np.max(abs(C.T@good))<1e-9


bad/good drift projection: -1.0000000000000007 -1.3522516439934406e-15


In [2]:
rng=np.random.default_rng(1414)
sources=np.column_stack([rng.laplace(size=3000),rng.uniform(-2,2,3000)])
mixing=np.array([[1.,.7],[.2,1.]])
mixed=sources@mixing.T
ica=FastICA(2,random_state=14,whiten='unit-variance',max_iter=1000)
recovered=ica.fit_transform(mixed)
corr=np.abs(np.corrcoef(sources.T,recovered.T)[:2,2:])
print('absolute source/component correlations:',corr)
assert np.all(corr.max(axis=1)>.95)
assert np.allclose(ica.inverse_transform(recovered),mixed)
print('Components remain unlabeled: separation is not artifact identification.')


absolute source/component correlations: [[0.9995486  0.03004309]
 [0.01225791 0.99992487]]
Components remain unlabeled: separation is not artifact identification.


## Check and explain

The correct residual matches the planted remaining source and has near-zero projection on the modeled drift. The bad sequence has drift coefficient−1. ICA recovers the two synthetic sources up to sign/permutation, and reconstruction matches the mixtures, but neither check assigns biological meaning.

## Deliberately wrong method

Regressing an unfiltered confound after filtering only y can bring unwanted variation back. Removing “component1” merely because it is first confuses decomposition order with classification. Deleting every confounds-table column can overfit or remove scientifically relevant variation.

## Transfer to an actual dataset or tool — guided assignment

Read DartBrains ICA and inspect one actual component’s spatial map, time course and spectrum using a preprocessed teaching run. In the fMRIPrep outputs, choose a documented subset of confounds and align rows to the BOLD volume indices. Record treatment of non-steady-state frames, drift terms, missing values and censoring. Present component-selection evidence and a sensitivity comparison; the complete real denoising strategy is an external supervised assignment.

**Submit:** a transformation card, one labeled figure or numerical result, the failed-method diagnosis, and the upstream-practice evidence. If the external exercise has not been run, mark it **not executed** and state the missing software/data; do not convert a proposed command into a claimed result.

## Exit questions and answer key

1. What does orthogonality establish? **Answer:** the specified least-squares component was removed numerically, not that the residual is artifact-free.
2. Why can ICA signs differ across runs? **Answer:** sign and order are not identifiable labels; inspect matched components and their mixing instead.


### Return to the research question

Revisit [PP02](../../curriculum/papers/processing.md#pp02), [PD01](../../curriculum/papers/design.md#pd01) and your initial prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure or section locator. Which part of the published result remains open after this exercise?
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Include this entry in the A2 portfolio when relevant.
